# Lab 11: Logistic Regression 2

รายวิชา `229351: Statistical Learning for Data Science 1`

ครึ่งแรกของ lab ประเมิน model Framingham จากคาบที่ 10 ให้ถูกวิธี: confusion matrix, train/test split, threshold และ ROC/AUC ครึ่งหลังใช้ข้อมูลรถ `Auto` (ISLR) ทำนายแหล่งผลิตสามกลุ่มด้วย multinomial logistic regression ตัวเลขทุกตัวตรงกับสไลด์ Lec 11 ถ้าค่าที่ได้ไม่ตรง แปลว่ามีขั้นตอนใดขั้นตอนหนึ่งผิด ไม่ใช่เรื่องของการปัดเศษ

Lab นี้เขียนตัวชี้วัดและ ROC ด้วย numpy ก่อน แล้วจึงใช้ scikit-learn เป็นตัวตรวจคำตอบ ไม่ใช่ในทางกลับกัน

## เป้าหมาย

1. แบ่ง train/test แบบ stratify และ standardize ด้วยค่าจาก train เท่านั้น
2. สร้าง confusion matrix และคำนวณ accuracy, recall, specificity, precision และ $F_1$ ด้วย numpy
3. อธิบายว่าทำไม accuracy จึงหลอกได้เมื่อกลุ่มไม่สมดุล
4. เลื่อน threshold และอธิบาย trade-off ระหว่าง recall กับ precision
5. เลือก threshold จากต้นทุนของความผิดแต่ละแบบ ไม่ใช่จาก test set
6. สร้าง ROC curve และคำนวณ AUC ด้วยมือ และอธิบาย AUC ในรูปของคู่ผู้ป่วยกับคนไม่ป่วย
7. หลีกเลี่ยงกับดัก AUC ที่คำนวณจาก `predict()` แทน `predict_proba()`
8. fit multinomial logistic regression ด้วย `statsmodels` ตีความสัมประสิทธิ์เทียบกับ baseline และคำนวณ softmax ด้วยมือ
9. เชื่อมสัมประสิทธิ์ของ scikit-learn กับของ `statsmodels`
10. ประเมิน classifier สามกลุ่มด้วย confusion matrix $3 \times 3$ และรายงานผลอย่างซื่อสัตย์

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

pd.set_option("display.float_format", lambda value: f"{value:.6g}")
np.set_printoptions(suppress=True)
plt.rcParams["figure.figsize"] = (9, 3.5)
plt.rcParams["axes.grid"] = True

# ข้อมูลทั้งสองไฟล์ถูก pin ไว้ที่ commit หนึ่งโดยเฉพาะ เพื่อให้คำตอบเชิงตัวเลขไม่เปลี่ยนตามเวลา
FRAMINGHAM_URL = "https://raw.githubusercontent.com/skitimo/ds351-data/6cf61a33e9eff25baef10cbff74a2856925db04a/framingham.csv"
AUTO_URL = "https://raw.githubusercontent.com/intro-stat-learning/ISLP_labs/ba2f7b8adc4a0a516db49d9f49f29327ee16a991/Auto.csv"

## 1. ข้อมูล Framingham และการแบ่ง train/test (8 คะแนน)

target คือ `TenYearCHD` (1 = เป็นโรคหลอดเลือดหัวใจภายใน 10 ปี) ใช้ predictor ครบ 15 ตัว และตัดแถวที่มีค่าหายแบบเดียวกับ Lab 10

กติกาการแบ่งข้อมูล

- `test_size=0.25`, `stratify=y` และ `random_state=0` เพื่อให้ทุกคนได้ split เดียวกัน
- `StandardScaler` ต้อง **fit บน train เท่านั้น** แล้วใช้ค่าเฉลี่ยและ SD ชุดนั้นแปลงทั้ง train และ test

In [ ]:
fram = pd.read_csv(FRAMINGHAM_URL).dropna()
y = fram["TenYearCHD"]
X = fram.drop(columns="TenYearCHD")
print(X.shape, f"CHD rate {y.mean():.4f}")

In [ ]:
# TODO: แบ่ง X, y เป็น train/test ตามกติกาข้างบน
#       แล้ว fit StandardScaler บน X_tr และแปลงทั้งสองส่วนเป็น Z_tr, Z_te
X_tr, X_te, y_tr, y_te = ...
scaler = ...
Z_tr = ...
Z_te = ...

print(f"train {len(y_tr)} rows, {int(y_tr.sum())} CHD   test {len(y_te)} rows, {int(y_te.sum())} CHD")

In [ ]:
assert (len(y_tr), int(y_tr.sum()), len(y_te), int(y_te.sum())) == (2743, 418, 915, 139)
assert np.allclose(Z_tr.mean(axis=0), 0, atol=1e-10), "Z_tr should have mean 0 in every column"
assert np.abs(Z_te.mean(axis=0)).max() > 1e-3, "Z_te must be scaled with the TRAIN mean and sd, not its own"
print("data OK")

เซลล์ข้างล่าง fit model เดียวกับคาบที่ 10 บน **train** แล้วเก็บความน่าจะเป็นของทั้งสองส่วนไว้ใช้ทั้ง lab

`tol=1e-8` ทำให้คำตอบตรงกับ `statsmodels` (ค่าเริ่มต้น `1e-4` หยุดเร็วไป ความน่าจะเป็นต่างกันได้ถึง 0.0016)

In [ ]:
model = LogisticRegression(C=np.inf, tol=1e-8, max_iter=10_000).fit(Z_tr, y_tr)
p_tr = model.predict_proba(Z_tr)[:, 1]     # P(CHD) for each training row
p_te = model.predict_proba(Z_te)[:, 1]     # P(CHD) for each test row
print(f"mean p-hat on train {p_tr.mean():.4f}  (train CHD rate {y_tr.mean():.4f})")

## Exercise 1: Confusion matrix และตัวชี้วัด (14 คะแนน)

ทาย $\hat{y} = 1$ เมื่อ $\hat{p} > t$ แล้วนับสี่ช่อง โดยใช้ layout ของ scikit-learn: **แถว = ค่าจริง คอลัมน์ = ค่าที่ทาย** เรียง 0 ก่อน 1

| | $\hat{y} = 0$ | $\hat{y} = 1$ |
|---|---|---|
| $y = 0$ | TN | FP |
| $y = 1$ | FN | TP |

$$\text{recall} = \frac{TP}{TP + FN}, \quad \text{specificity} = \frac{TN}{TN + FP}, \quad \text{precision} = \frac{TP}{TP + FP}, \quad F_1 = \frac{2 \cdot \text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$$

เขียนสองฟังก์ชันด้วย numpy **ห้ามใช้ `sklearn.metrics`** ในสองฟังก์ชันนี้

In [ ]:
# TODO: เขียน confusion_counts(y_true, p, t) ที่คืน (tn, fp, fn, tp) เป็น int เมื่อทาย 1 ถ้า p > t
#       และ metrics(tn, fp, fn, tp) ที่คืน dict ของ accuracy, recall, specificity, precision, f1
#       แล้วคำนวณ c_test, m_test ที่ t = 0.5 บน test set และ baseline_te = accuracy ของการทาย 0 ทุกคน
def confusion_counts(y_true, p, t):
    y_true = np.asarray(y_true)
    y_hat = ...
    tn = ...
    fp = ...
    fn = ...
    tp = ...
    return tn, fp, fn, tp


def metrics(tn, fp, fn, tp):
    precision = ...    # ระวังกรณี tp + fp == 0
    recall = ...
    return {
        "accuracy": ...,
        "recall": recall,
        "specificity": ...,
        "precision": precision,
        "f1": ...,
    }


c_test = confusion_counts(y_te, p_te, 0.5)
m_test = metrics(*c_test)
baseline_te = ...
print("tn, fp, fn, tp =", c_test)
print({name: round(value, 4) for name, value in m_test.items()}, f"baseline {baseline_te:.4f}")

In [ ]:
# your function must agree with scikit-learn's -- and with its layout
assert c_test == tuple(int(v) for v in confusion_matrix(y_te, (p_te > 0.5).astype(int)).ravel())
assert c_test == (774, 2, 131, 8)
assert round(m_test["accuracy"], 4) == 0.8546 and round(baseline_te, 4) == 0.8481
assert round(m_test["recall"], 4) == 0.0576 and round(m_test["specificity"], 4) == 0.9974
assert round(m_test["precision"], 4) == 0.8000 and round(m_test["f1"], 4) == 0.1074
print("Exercise 1 OK")

In [ ]:
print(confusion_matrix(y_te, (p_te > 0.5).astype(int)))   # rows = actual, columns = predicted

**เขียนคำตอบข้อ 1** (ตอบให้ครบทั้งสี่ข้อ)

1. accuracy ที่ได้ดีไหม เทียบกับ `baseline_te` และตัวชี้วัดตัวไหนบอกความจริงเกี่ยวกับ model นี้มากที่สุด
2. อธิบาย recall และ precision ที่ได้เป็นภาษาที่แพทย์เข้าใจ โดยใช้จำนวนคนจริง
3. ในงานคัดกรองโรคหัวใจ ความผิดแบบไหนแพงกว่า FP หรือ FN เพราะอะไร
4. ISLR Table 4.4 วางตาราง **แถว = ค่าที่ทาย** คอลัมน์ = ค่าจริง ใน layout นั้น เลข 131 และเลข 2 จะอยู่ช่องไหน และถ้าอ่าน `cm[0, 1]` เป็น FN โดยไม่ดูหัวตาราง จะได้ recall เท่าใด

_(เขียนคำตอบที่นี่)_

## Exercise 2: Train กับ test และ `stratify` (14 คะแนน)

1. ใช้ `confusion_counts` และ `metrics` กับ **train set** ที่ t = 0.5 แล้วสร้างตาราง `compare` ที่มีสองคอลัมน์ `train` และ `test`
2. แบ่งข้อมูลอีกครั้งด้วย `random_state=0` และ `test_size=0.25` เท่าเดิม แต่ **ไม่ใส่** `stratify` แล้วดูสัดส่วนผู้ป่วยใน train และ test

In [ ]:
# TODO: m_train, compare, และ y_tr_u, y_te_u จากการแบ่งแบบไม่ stratify
m_train = ...
compare = ...       # pd.DataFrame ที่มีคอลัมน์ "train" และ "test"
_, _, y_tr_u, y_te_u = ...

print(compare.round(4))
print(f"without stratify: train CHD rate {y_tr_u.mean():.4f}, test CHD rate {y_te_u.mean():.4f} ({int(y_te_u.sum())} cases)")

In [ ]:
assert list(compare.columns) == ["train", "test"]
assert round(m_train["accuracy"], 4) == 0.8542 and round(m_train["recall"], 4) == 0.0766
assert int(y_te_u.sum()) == 147 and round(y_te_u.mean(), 4) == 0.1607 and round(y_tr_u.mean(), 4) == 0.1495
print("Exercise 2 OK")

**เขียนคำตอบข้อ 2** (ตอบให้ครบทั้งสามข้อ)

1. ตัวเลขบน train กับ test ใกล้กันมาก แปลว่าอะไร และแปลว่า model ดีแล้วหรือไม่
2. ทำไมต้อง fit `StandardScaler` บน train เท่านั้น ถ้า fit บนข้อมูลทั้งหมดจะเกิดอะไรขึ้น
3. การแบ่งแบบไม่ stratify ได้สัดส่วนผู้ป่วยใน test เท่าใด ทำไมจึงเป็นปัญหาต่อการรายงานผล

_(เขียนคำตอบที่นี่)_

## Exercise 3: Threshold และต้นทุน (18 คะแนน)

### 3.1 เลื่อน threshold บน test set

สร้างตาราง `sweep` ที่มี index เป็น $t \in \{0.5, 0.3, 0.2, 0.1\}$ และคอลัมน์ `pred_pos` (จำนวนที่ทายว่าเป็นโรค), `TP`, `FP`, `recall`, `precision`, `accuracy`

In [ ]:
# TODO: สร้าง sweep ด้วย confusion_counts และ metrics
rows = []
for t in [0.5, 0.3, 0.2, 0.1]:
    ...   # ต่อ dict หนึ่งแถวเข้า rows ด้วย rows.append({...})
sweep = pd.DataFrame(rows).set_index("t")
print(sweep.round(4))

In [ ]:
assert list(sweep.index) == [0.5, 0.3, 0.2, 0.1]
assert sweep.loc[0.5, "TP"] == 8 and sweep.loc[0.1, "TP"] == 119 and sweep.loc[0.2, "FP"] == 146
assert sweep.loc[0.3, "pred_pos"] == 95 and round(sweep.loc[0.3, "recall"], 4) == 0.2806
assert round(sweep.loc[0.1, "precision"], 4) == 0.2297 and round(sweep.loc[0.2, "accuracy"], 4) == 0.7650
print("3.1 OK")

In [ ]:
# the whole trade-off on the test set
ts = np.linspace(0.02, 0.7, 137)
curves = pd.DataFrame([metrics(*confusion_counts(y_te, p_te, t)) for t in ts], index=ts)
ax = curves[["recall", "specificity", "precision", "accuracy"]].plot()
ax.axvline(0.5, color="gray", linestyle="--")
ax.set_xlabel("threshold t"); ax.set_ylim(0, 1.02)
plt.show()

### 3.2 เลือก threshold จากต้นทุน

สมมติว่าการพลาดผู้ป่วยหนึ่งคน (FN) แย่เท่ากับ false alarm ห้าครั้ง (FP): $c_{FP} = 1$, $c_{FN} = 5$ ต้นทุนรวม $= c_{FP} \cdot FP + c_{FN} \cdot FN$

1. `t_cost`: threshold จากสูตรในสไลด์ $t = c_{FP} / (c_{FP} + c_{FN})$
2. `t_train`: threshold บน `grid` ที่ทำให้ต้นทุนรวม **บน train** ต่ำที่สุด (ถ้าเสมอกันให้เลือกค่าแรก `np.argmin` ทำให้อยู่แล้ว)
3. `test_costs`: dict ของต้นทุนรวมบน **test** ที่ t = 0.5, `t_cost` และ `t_train`

In [ ]:
# TODO: total_cost, t_cost, t_train และ test_costs
C_FP, C_FN = 1, 5
grid = np.round(np.arange(0.01, 1.0, 0.01), 2)


def total_cost(y_true, p, t):
    ...
    return ...


t_cost = ...
train_costs = ...   # np.array ของต้นทุนบน train ทีละค่าใน grid
t_train = ...
test_costs = ...    # {0.5: ..., t_cost: ..., t_train: ...}

print(f"t_cost = {t_cost:.4f}   t_train = {t_train:.2f}")
print({f"{t:.4f}": cost for t, cost in test_costs.items()})
print("test at t_cost: tn, fp, fn, tp =", confusion_counts(y_te, p_te, t_cost))

In [ ]:
assert abs(t_cost - 1 / 6) < 1e-12
assert t_train == 0.17 and int(train_costs.min()) == 1420
assert test_costs[0.5] == 657 and test_costs[t_cost] == 440 and test_costs[t_train] == 442
assert confusion_counts(y_te, p_te, t_cost) == (571, 205, 47, 92)
print("3.2 OK")

**เขียนคำตอบข้อ 3** (ตอบให้ครบทั้งสามข้อ)

1. ถ้าใช้ model นี้คัดกรองผู้ป่วย คุณจะแนะนำ threshold เท่าใด อ้างจำนวนผู้ป่วยที่จับได้ จำนวน false alarm และต้นทุนรวม
2. `t_train` กับ `t_cost` ใกล้กันหรือไม่ ทำไมสองวิธีที่ต่างกันจึงให้คำตอบใกล้กัน
3. ทำไมจึงไม่ควรเลือก threshold โดยลองบน test set

_(เขียนคำตอบที่นี่)_

## Exercise 4: ROC curve, AUC และกับดัก (16 คะแนน)

### 4.1 ROC และ AUC ด้วย numpy

เรียงคนใน test set จาก `p_te` มากไปน้อย แล้วลด threshold ลงทีละคน หลังจากนับ $k$ คนแรกเป็น positive

- `tpr[k]` = สัดส่วนของผู้ป่วยทั้งหมดที่อยู่ใน $k$ คนแรก
- `fpr[k]` = สัดส่วนของคนไม่ป่วยทั้งหมดที่อยู่ใน $k$ คนแรก

สำหรับ $k = 0, 1, \ldots, n$ (`tpr[0] = fpr[0] = 0`) แล้วหา AUC สองวิธี

1. `auc_trap`: พื้นที่ใต้กราฟด้วยกฎสี่เหลี่ยมคางหมู `np.trapezoid(tpr, fpr)` (numpy รุ่นก่อน 2.0 ใช้ `np.trapz`)
2. `auc_pairs`: สัดส่วนของคู่ (ผู้ป่วย, คนไม่ป่วย) ทั้งหมด ที่ผู้ป่วยได้ $\hat{p}$ สูงกว่า

In [ ]:
# TODO: tpr, fpr, auc_trap และ auc_pairs
order = np.argsort(-p_te, kind="stable")
y_sorted = y_te.to_numpy()[order]
tpr = ...
fpr = ...
auc_trap = ...

pos = ...    # p_te ของผู้ป่วย
neg = ...    # p_te ของคนไม่ป่วย
auc_pairs = ...
print(f"AUC: trapezoid {auc_trap:.4f}   pairs {auc_pairs:.4f}   sklearn {roc_auc_score(y_te, p_te):.4f}")

In [ ]:
assert len(tpr) == len(y_te) + 1 and tpr[0] == 0 and fpr[0] == 0 and tpr[-1] == 1 and fpr[-1] == 1
assert abs(auc_trap - roc_auc_score(y_te, p_te)) < 1e-12
assert abs(auc_pairs - roc_auc_score(y_te, p_te)) < 1e-12
assert round(auc_trap, 4) == 0.7510 and len(pos) * len(neg) == 107864
print("4.1 OK")

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.plot(fpr, tpr, label=f"logistic model, AUC = {auc_trap:.4f}")
ax.plot([0, 1], [0, 1], "--", color="gray", label="random guessing")
for t in [0.5, 0.3, 0.2, 0.1]:
    m = metrics(*confusion_counts(y_te, p_te, t))
    ax.plot(1 - m["specificity"], m["recall"], "o", color="crimson")
    ax.annotate(f"t = {t}", (1 - m["specificity"], m["recall"]), xytext=(6, -10), textcoords="offset points")
ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate (recall)"); ax.legend(loc="lower right")
plt.show()

### 4.2 กับดัก: AUC จากป้ายชื่อ

เซลล์ข้างล่างคำนวณ AUC แบบที่ **ผิด** สองแบบโดยตั้งใจ ทั้งสองแบบไม่ error ไม่เตือน ดูผลลัพธ์ก่อน แล้วแสดงในเซลล์ถัดไปว่าแต่ละค่ามาจากไหน

In [ ]:
wrong_labels = roc_auc_score(y_te, model.predict(Z_te))              # 0/1 labels instead of scores
wrong_column = roc_auc_score(y_te, model.predict_proba(Z_te)[:, 0])   # P(y = 0) instead of P(y = 1)
print(f"from predict(): {wrong_labels:.4f}   from predict_proba[:, 0]: {wrong_column:.4f}   correct: {auc_trap:.4f}")

In [ ]:
# TODO: explain_labels = (recall + specificity) / 2 ที่ t = 0.5 จาก m_test
#       explain_column = 1 - auc_trap
explain_labels = ...
explain_column = ...
print(f"{explain_labels:.4f}  {explain_column:.4f}")

In [ ]:
assert round(wrong_labels, 4) == 0.5275 and round(wrong_column, 4) == 0.2490
assert abs(explain_labels - wrong_labels) < 1e-12 and abs(explain_column - wrong_column) < 1e-12
print("4.2 OK")

**เขียนคำตอบข้อ 4** (ตอบให้ครบทั้งสามข้อ)

1. อธิบายความหมายของ AUC $0.7510$ ด้วยคู่ผู้ป่วยกับคนไม่ป่วย
2. ทำไม `roc_auc_score(y_te, model.predict(Z_te))` จึงได้ 0.5275 คนที่ไม่รู้กับดักนี้จะสรุปเกี่ยวกับ model ผิดอย่างไร และ `[:, 0]` ผิดอย่างไร
3. AUC ไม่ขึ้นกับ threshold ข้อนี้ทำให้ AUC ใช้ทำอะไรได้ และใช้ทำอะไรไม่ได้

_(เขียนคำตอบที่นี่)_

## Exercise 5: Multinomial logistic regression (16 คะแนน)

ข้อมูล `Auto` (ISLR) 392 คัน ชุดเดียวกับ Lab 05 คราวนี้ target คือ `origin` ซึ่งเป็นหมวดหมู่สามกลุ่มที่เก็บเป็นเลข: 1 = American, 2 = European, 3 = Japanese

- `disp100` = `displacement / 100` ขนาดเครื่องยนต์ในหน่วย 100 ลูกบาศก์นิ้ว
- `year` = ปีรุ่น 70 ถึง 82 (ค.ศ. 1970 ถึง 1982)

model คือ baseline-category logit โดยมี American เป็น baseline

$$\log\frac{P(Y = k \mid x)}{P(Y = 1 \mid x)} = \beta_{0k} + \beta_{1k}\,\texttt{disp100} + \beta_{2k}\,\texttt{year}, \qquad k = 2, 3$$

In [ ]:
auto = pd.read_csv(AUTO_URL)
auto["disp100"] = auto["displacement"] / 100
print(auto.shape)
print(auto["origin"].value_counts().sort_index())

In [ ]:
# TODO: fit sm.MNLogit ของ origin บน disp100 และ year (อย่าลืม add_constant) เก็บ X_auto และ mn
X_auto = ...
mn = ...
print(mn.summary())

In [ ]:
assert list(mn.params.columns) == [0, 1], "column 0 = European vs American, column 1 = Japanese vs American"
assert np.allclose(mn.params.to_numpy().round(4), [[14.3803, 5.4989], [-3.6485, -4.9855], [-0.1331, 0.0033]])
assert round(mn.pvalues.loc["year", 1], 4) == 0.9479
print("5.1 OK")

### 5.2 Softmax ด้วยมือ

ให้ $\eta_1 = 0$ (baseline) และ $\eta_k = \beta_{0k} + \beta_{1k}\,\texttt{disp100} + \beta_{2k}\,\texttt{year}$ สำหรับ $k = 2, 3$

$$P(Y = k \mid x) = \frac{e^{\eta_k}}{e^{\eta_1} + e^{\eta_2} + e^{\eta_3}}$$

1. `softmax_probs(params, disp100, year)` คืน array ความน่าจะเป็น `[American, European, Japanese]` จาก `mn.params`
2. `p_car`: รถเครื่อง 100 ลูกบาศก์นิ้ว (`disp100 = 1`) ปีรุ่น 1980 (`year = 80`)
3. `odds_mult`: $e^{\beta}$ ของ `disp100` ทั้งสองคอลัมน์ (ตัวคูณ odds เทียบ American เมื่อเครื่องใหญ่ขึ้น 100 ลูกบาศก์นิ้ว)
4. `jap_vs_eur`: ตัวคูณ odds ของ Japanese **เทียบกับ European** เมื่อเครื่องใหญ่ขึ้น 100 ลูกบาศก์นิ้ว โดยไม่ fit ใหม่

In [ ]:
# TODO: softmax_probs, p_car, odds_mult และ jap_vs_eur
def softmax_probs(params, disp100, year):
    x = np.array([1.0, disp100, year])
    eta = ...    # [0, eta_European, eta_Japanese] จาก params[0] และ params[1]
    ...
    return ...


p_car = ...
odds_mult = ...
jap_vs_eur = ...
print("P(American, European, Japanese) =", p_car.round(4))
print("odds multipliers per +100 cu in:", odds_mult.round(4).tolist(), f"  Japanese vs European: {jap_vs_eur:.4f}")

In [ ]:
assert np.allclose(p_car, np.asarray(mn.predict(np.array([[1.0, 1.0, 80]])))[0])
assert np.allclose(p_car.round(4), [0.2350, 0.2547, 0.5103]) and abs(p_car.sum() - 1) < 1e-12
assert np.allclose(np.asarray(odds_mult).round(4), [0.0260, 0.0068]) and round(jap_vs_eur, 4) == 0.2626
print("5.2 OK")

In [ ]:
# (a) three separate binary fits do not give probabilities that add up
ovr = np.column_stack([
    sm.Logit((auto["origin"] == k).astype(int), X_auto).fit(disp=0).predict(X_auto) for k in (1, 2, 3)
])
row_sums = ovr.sum(axis=1)
print(f"three binary fits: row sums {row_sums.min():.4f} .. {row_sums.max():.4f}")
print(f"MNLogit:           row sums {np.asarray(mn.predict(X_auto)).sum(axis=1).min():.4f} .. "
      f"{np.asarray(mn.predict(X_auto)).sum(axis=1).max():.4f}")

# (b) P(European) in 1972 as the engine grows
for d in [0.7, 1.0, 1.3, 1.6]:
    print(f"1972, {d * 100:.0f} cu in: P(European) = {softmax_probs(mn.params, d, 72)[1]:.4f}")

**เขียนคำตอบข้อ 5** (ตอบให้ครบทั้งสี่ข้อ)

1. ตีความ `odds_mult` ตัวแรก ($e^{\beta}$ ของ `disp100` ในสมการ European) ให้ถูกสเกล และอธิบายว่า `jap_vs_eur` คำนวณได้โดยไม่ fit ใหม่เพราะอะไร
2. `year` ในสมการ Japanese มี $p$-value เท่าใด แปลว่าอะไร
3. รถใน `p_car` ถูกทายว่ามาจากไหน มั่นใจแค่ไหน และทำไม $P(\text{European})$ ในปี 1972 จึงเพิ่มขึ้นเมื่อเครื่องโตจาก 70 เป็น 100 ลูกบาศก์นิ้ว ทั้งที่ $\beta$ ของ European ติดลบ
4. ทำไมการ fit binary logistic regression สามตัวแยกกันจึงใช้แทน multinomial ไม่ได้ อ้างตัวเลขจากเซลล์ข้างบน

_(เขียนคำตอบที่นี่)_

## Exercise 6: scikit-learn และ confusion matrix สามกลุ่ม (14 คะแนน)

### 6.1 สัมประสิทธิ์ของ scikit-learn

`LogisticRegression` ใช้ softmax อัตโนมัติเมื่อ target มีมากกว่าสองกลุ่ม แต่ **ไม่มี baseline**: `coef_` มีหนึ่งแถวต่อหนึ่งกลุ่ม

1. fit `LogisticRegression(C=np.inf, tol=1e-8, max_iter=10_000)` บน `auto[["disp100", "year"]]` ทั้ง 392 คัน เป็น `sk_auto`
2. `coef_vs_american` และ `intercept_vs_american`: ลบแถวของ American (แถวแรก) ออกจากทุกแถว

In [ ]:
# TODO: sk_auto, coef_vs_american และ intercept_vs_american
sk_auto = ...
coef_vs_american = ...
intercept_vs_american = ...
print(sk_auto.coef_.round(4))
print(coef_vs_american.round(4), intercept_vs_american.round(4))

In [ ]:
assert sk_auto.coef_.shape == (3, 2) and np.allclose(sk_auto.coef_.sum(axis=0), 0, atol=1e-8)
assert np.allclose(coef_vs_american[1:].T, mn.params.loc[["disp100", "year"]].to_numpy(), atol=1e-4)
assert np.allclose(intercept_vs_american[1:], mn.params.loc["const"].to_numpy(), atol=1e-3)
assert np.allclose(sk_auto.predict_proba(auto[["disp100", "year"]]), np.asarray(mn.predict(X_auto)), atol=1e-6)
print("6.1 OK")

### 6.2 ประเมินบน test set

1. แบ่ง `auto` เป็น `auto_tr`, `auto_te` แบบ 75/25 `stratify=auto["origin"]` และ `random_state=0`
2. fit model เดียวกันบน `auto_tr` ทาย `auto_te` แล้วสร้าง `cm3 = confusion_matrix(..., labels=[1, 2, 3])`
3. `recall_by_class`, `precision_by_class` และ `accuracy_auto` จาก `cm3` ด้วย numpy (แนวทแยงหารผลรวมแถว หรือผลรวมคอลัมน์)

In [ ]:
# TODO: auto_tr, auto_te, cm3, recall_by_class, precision_by_class, accuracy_auto
auto_tr, auto_te = ...
sk_tr = ...
cm3 = ...
recall_by_class = ...
precision_by_class = ...
accuracy_auto = ...

print(cm3)
print("recall", recall_by_class.round(4), " precision", precision_by_class.round(4))
print(f"accuracy {accuracy_auto:.4f}   always American {(auto_te['origin'] == 1).mean():.4f}")

In [ ]:
assert cm3.tolist() == [[53, 3, 5], [6, 6, 5], [5, 3, 12]]
assert np.allclose(recall_by_class.round(4), [0.8689, 0.3529, 0.6000])
assert np.allclose(precision_by_class.round(4), [0.8281, 0.5000, 0.5455])
assert round(accuracy_auto, 4) == 0.7245 and round((auto_te["origin"] == 1).mean(), 4) == 0.6224
print("6.2 OK")

**เขียนคำตอบข้อ 6** (ตอบให้ครบทั้งสามข้อ)

1. ทำไม `coef_` ของ scikit-learn แต่ละคอลัมน์จึงรวมกันได้ศูนย์ และทำไมจึงอ่านแต่ละแถวแบบเดียวกับ `mn.params` ไม่ได้
2. รายงานผลบน test set อย่างซื่อสัตย์: model ทำอะไรได้ดี ทำอะไรไม่ได้ อ้างตัวเลขจาก `cm3`
3. test set มีรถยุโรปกี่คัน ตัวเลข recall ของกลุ่มนี้เชื่อถือได้แค่ไหน

_(เขียนคำตอบที่นี่)_

## โจทย์เสริม (ไม่คิดคะแนน)

1. ทำซ้ำการแบ่ง Framingham ด้วย `random_state` 0 ถึง 99 (stratify เหมือนเดิม) แล้วเก็บ test AUC บนสไลด์ได้ต่ำสุด $0.6853$ มัธยฐาน $0.7303$ สูงสุด $0.7723$ ของคุณได้เท่าใด split ที่เราใช้ทั้ง lab อยู่ตรงไหน
2. เพิ่ม `mpg` และ `weight` เข้าไปใน multinomial model สัมประสิทธิ์ของ `weight` ต่อ 1000 ปอนด์ในสมการ European เป็นบวก ($6.7293$) ทั้งที่รถยุโรปเบากว่ารถอเมริกันโดยเฉลี่ย (2433 กับ 3372 ปอนด์) อธิบายด้วยเนื้อหาคาบที่ 6
3. เลือก threshold ที่ทำให้ $F_1$ บน train สูงสุด แล้วรายงาน recall และ precision บน test เทียบกับ `t_cost`

## Checklist ก่อนส่ง

- [ ] train 2,743 แถว (CHD 418) test 915 แถว (CHD 139) และ `Z_te` ใช้ค่าเฉลี่ยกับ SD ของ train
- [ ] `confusion_counts` ได้ `(774, 2, 131, 8)` ตรงกับ `confusion_matrix(...).ravel()`
- [ ] recall `0.0576` precision `0.8000` accuracy `0.8546` เทียบกับ baseline `0.8481`
- [ ] train accuracy `0.8542` และการแบ่งแบบไม่ stratify ได้ผู้ป่วยใน test 147 คน
- [ ] ตาราง `sweep` ครบสี่ threshold และ `t_cost = 1/6` ได้ต้นทุน 440 เทียบกับ 657 ที่ 0.5
- [ ] `t_train = 0.17` เลือกจาก train เท่านั้น
- [ ] `auc_trap` และ `auc_pairs` ได้ `0.7510` ตรงกับ `roc_auc_score`
- [ ] อธิบายได้ว่า `0.5275` และ `0.2490` มาจากไหน
- [ ] `mn.params` ได้ $14.3803, -3.6485, -0.1331$ และ $5.4989, -4.9855, 0.0033$
- [ ] `p_car` ได้ `[0.2350, 0.2547, 0.5103]`
- [ ] `coef_vs_american` ตรงกับ `mn.params`
- [ ] `cm3` ได้ `[[53, 3, 5], [6, 6, 5], [5, 3, 12]]` และ accuracy `0.7245`
- [ ] ตอบคำถามทุกข้อในทุก markdown cell ที่ขึ้นต้นด้วย "เขียนคำตอบ"
- [ ] restart kernel แล้ว run all ผ่านตลอดโดยไม่มี error
- [ ] เปลี่ยนชื่อไฟล์เป็น `lab11_studentid.ipynb` ก่อนส่ง